In [0]:
%pip install google-cloud-storage

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../tests/test_silver_layer

In [0]:
%run ../utils/gcp_setup

In [0]:
import os
from google.cloud import storage
import pyspark.sql.functions as F
import logging
import sys

# 1. Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("TaxiBronzeIngestion")

**Databricks Widgets Set Up**

In [0]:
dbutils.widgets.text("credentials_path", "")
dbutils.widgets.text("gcs_bucket_name", "")
dbutils.widgets.text("gcs_folder_prefix_2025", "")
dbutils.widgets.text("gcs_folder_prefix_2026", "")
dbutils.widgets.text("dbfs_taxi_2025_path", "")
dbutils.widgets.text("dbfs_taxi_2026_path", "")
dbutils.widgets.text("bronze_taxi_2025", "")
dbutils.widgets.text("bronze_taxi_2026", "")


credentials_path = dbutils.widgets.get("credentials_path") or "/Workspace/Shared/service-account.json"
gcs_bucket_name = dbutils.widgets.get("gcs_bucket_name") or "prefect-bucket-latypov"
gcs_folder_prefix_2025 = dbutils.widgets.get("gcs_folder_prefix_2025") or "chicago_taxi_data/year=2025"
gcs_folder_prefix_2026 = dbutils.widgets.get("gcs_folder_prefix_2026") or "chicago_taxi_data/year=2026"
dbfs_taxi_2025_path = dbutils.widgets.get("dbfs_taxi_2025_path") or "/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025"
dbfs_taxi_2026_path = dbutils.widgets.get("dbfs_taxi_2026_path") or "/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2026"
bronze_taxi_2025 = dbutils.widgets.get("bronze_taxi_2025") or "chicago_taxi_data.bronze.bronze_taxi_2025"
bronze_taxi_2026 = dbutils.widgets.get("bronze_taxi_2026") or "chicago_taxi_data.bronze.bronze_taxi_2026"

**Google Cloud credentials Set Up**

In [0]:

# Set up Google Cloud service account keys
logger.info(f"Set up Google Cloud service account keys")
setup_gcp_creds(credentials_path=credentials_path)


2026-05-14 05:44:51 - INFO - Set up Google Cloud service account keys
2026-05-14 05:44:52 - INFO - Found service account file
2026-05-14 05:44:52 - INFO - Successfully authenticated. Found 2 buckets.


**Connect to GCS Chicago Taxi Data for 2025-2026 years**

In [0]:
try:
    # 1. Setup GCS client
    client = storage.Client()
    bucket = client.get_bucket(gcs_bucket_name)
    logger.info(f"GCS bucket: {bucket}")
    # 2. Get top-level "folders" under chicago_taxi_data
    top_level = bucket.list_blobs(prefix="chicago_taxi_data/", delimiter="/")
    list(top_level)  # force iteration to populate prefixes

    MONTHS_2025 = []
    MONTHS_2026 = []

    for prefix in top_level.prefixes:
        logger.info(f"Year folder: {prefix}")
        # 3. list subfolders (months) under each year
        year_blobs = bucket.list_blobs(prefix=prefix, delimiter="/")
        list(year_blobs)  # populate prefixes
        if "2025" in prefix:
            MONTHS_2025.extend(year_blobs.prefixes)
        elif "2026" in prefix:
            MONTHS_2026.extend(year_blobs.prefixes)

    logger.info(f"GCS {bucket}/{prefix} contains: {len(MONTHS_2025)} files")
    logger.info(f"2025 months: {MONTHS_2025}")
    logger.info(f"GCS {bucket}/{prefix} contains: {len(MONTHS_2026)} files")
    logger.info(f"2026 months: {MONTHS_2026}")
    try: 
        assert len(MONTHS_2025) <= 12 and len(MONTHS_2026) <= 12

    except AssertionError as e:
        print(f"AssertionError: {e}")
        dbutils.notebook.exit("AssertionError: Check your GCS bucket structure")
except Exception as e:
    logger.error(f"File ingestestion failed: {e}")
    dbutils.notebook.exit(f"File ingestestion failed: {e}")



2026-05-14 05:45:01 - INFO - GCS bucket: <Bucket: prefect-bucket-latypov>
2026-05-14 05:45:01 - INFO - Year folder: chicago_taxi_data/year=2026/
2026-05-14 05:45:02 - INFO - Year folder: chicago_taxi_data/year=2025/
2026-05-14 05:45:02 - INFO - GCS <Bucket: prefect-bucket-latypov>/chicago_taxi_data/year=2025/ contains: 12 files
2026-05-14 05:45:02 - INFO - 2025 months: ['chicago_taxi_data/year=2025/month=09/', 'chicago_taxi_data/year=2025/month=06/', 'chicago_taxi_data/year=2025/month=10/', 'chicago_taxi_data/year=2025/month=03/', 'chicago_taxi_data/year=2025/month=11/', 'chicago_taxi_data/year=2025/month=12/', 'chicago_taxi_data/year=2025/month=08/', 'chicago_taxi_data/year=2025/month=02/', 'chicago_taxi_data/year=2025/month=07/', 'chicago_taxi_data/year=2025/month=04/', 'chicago_taxi_data/year=2025/month=01/', 'chicago_taxi_data/year=2025/month=05/']
2026-05-14 05:45:02 - INFO - GCS <Bucket: prefect-bucket-latypov>/chicago_taxi_data/year=2025/ contains: 3 files
2026-05-14 05:45:02 - 

**Chicago Taxi Data 2025-2026 years Ingestion to Databricks Unity Catalog Volume**

In [0]:

def download_to_databricks(months_amount, gcs_path, dbfs_dir):
    os.makedirs(dbfs_dir, exist_ok=True)
    logger.info(f"Starting download from GCS to {dbfs_dir}...")
    for i in range(1, months_amount+1):
        month_str = f"month={i:02d}"
        prefix = f"{gcs_path}/{month_str}/"
        
        gcs_files = list(bucket.list_blobs(prefix=prefix))

        if not gcs_files:
            continue

        local_dbfs_dir = os.path.join(f"{dbfs_dir}/{month_str}")
        os.makedirs(local_dbfs_dir, exist_ok=True)

        for blob in gcs_files:
            if blob.name.endswith(".parquet"):
                file_name = blob.name.split("/")[-1]
                destination_path = os.path.join(local_dbfs_dir, file_name)
                
                blob.download_to_filename(destination_path)
                logger.info(f"Downloaded: {blob.name} from GCS to {destination_path}")


In [0]:
run_2025 = download_to_databricks(months_amount = len(MONTHS_2025), gcs_path=gcs_folder_prefix_2025, dbfs_dir=dbfs_taxi_2025_path)
run_2026 = download_to_databricks(months_amount = len(MONTHS_2026), gcs_path=gcs_folder_prefix_2026, dbfs_dir=dbfs_taxi_2026_path)
logger.info(f"Starting Ingesting data from GCS to Databricks")
print(run_2025)
print(run_2026)
logger.info(f"Starting Ingesting data from GCS to Databricks")

2026-05-13 12:05:22 - INFO - Starting download from GCS to /Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025...
2026-05-13 12:05:24 - INFO - Downloaded: chicago_taxi_data/year=2025/month=01/part_2220000_b626f0fd.parquet from GCS to /Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=01/part_2220000_b626f0fd.parquet
2026-05-13 12:05:25 - INFO - Downloaded: chicago_taxi_data/year=2025/month=01/part_2250000_6ee525cb.parquet from GCS to /Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=01/part_2250000_6ee525cb.parquet
2026-05-13 12:05:26 - INFO - Downloaded: chicago_taxi_data/year=2025/month=01/part_2280000_0007766c.parquet from GCS to /Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=01/part_2280000_0007766c.parquet
2026-05-13 12:05:27 - INFO - Downloaded: chicago_taxi_data/year=2025/month=01/part_2310000_d94bbbb0.parquet from GCS to /Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=01/pa

None
None


In [0]:
df_raw_taxi_2025 = spark.read.parquet(dbfs_taxi_2025_path)
df_raw_taxi_2026 = spark.read.parquet(dbfs_taxi_2026_path)


required_columns = ["trip_start_timestamp", "trip_id", "fare"]
validate_schema(df_raw_taxi_2025, required_columns)
validate_schema(df_raw_taxi_2026, required_columns)


df_bronze_taxi_2025 = df_raw_taxi_2025.withColumn("ingestion_timestamp", F.current_timestamp()).\
                                    withColumn("source_file", F.col("_metadata.file_path"))


df_bronze_taxi_2026 = df_raw_taxi_2026.withColumn("ingestion_timestamp", F.current_timestamp()).\
                                    withColumn("source_file", F.col("_metadata.file_path"))      


df_bronze_taxi_2025.write.format("delta").mode("overwrite").saveAsTable(bronze_taxi_2025)

df_bronze_taxi_2026.write.format("delta").mode("overwrite").saveAsTable(bronze_taxi_2026)
                        

2026-05-14 05:45:25 - INFO - validate_schema - PASSED (All required columns present: ['trip_start_timestamp', 'trip_id', 'fare'])
2026-05-14 05:45:26 - INFO - validate_schema - PASSED (All required columns present: ['trip_start_timestamp', 'trip_id', 'fare'])


In [0]:
display(spark.sql(f"SELECT * FROM {bronze_taxi_2025} ORDER BY ingestion_timestamp DESC LIMIT 5"))

trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,year,month,ingestion_timestamp,source_file
4c2ab748f91ed577504cb81b7d9c039b0ec1c690,89a40bbeb44e22e8169b4ba823c5eee94b324219fbe8c007ec52243acb7b0aed5ac28fc53132e13d6deb4fcd6eec8911cc70ac7233fe3bdd8f8a5ea6f23e5f71,2025-03-31T23:00:00.000,2025-03-31T23:30:00.000,1109,5.4,null,null,28,6,17.57,2,0,0,20.07,Mobile,Tac - Yellow Cab Association,41.874005383,-87.66351755,"List(List(-87.6635175498, 41.874005383), Point)",41.944226601,-87.655998182,"List(List(-87.6559981815, 41.9442266014), Point)",2025,3,2026-05-14T05:45:29.752Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=03/part_1230000_50789e9a.parquet
43db219890bfd8418f155ceb8af57e16095a4b94,f7684402b7b01a62af891e65189e28fb61890f2ef5922b1d3d411329c1c7052673b1f781dda89f2fdcfa99aac10a3e65075093505297afef46c96a138bd3e502,2025-03-31T23:00:00.000,2025-03-31T23:15:00.000,960,4.5,null,null,2,null,15.25,0,0,0,15.25,Unknown,Taxi Affiliation Services,42.001571027,-87.695012589,"List(List(-87.6950125892, 42.001571027), Point)",null,null,null,2025,3,2026-05-14T05:45:29.752Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=03/part_1230000_50789e9a.parquet
5071b667cef8a6f9bb0f3503faa8e11cc7b1a77a,565d45f83a15ee7d65f7e519ebd063597b2e7e4ef6309038af538a0091ba887e72011276f3c1f3a051eb1194c6af5cff4e431e1628b6649b6092c43f7a48fcf1,2025-03-31T23:00:00.000,2025-03-31T23:15:00.000,728,3.02,null,null,32,33,11,2.9,0,3,17.4,Credit Card,Sun Taxi,41.878865584,-87.625192142,"List(List(-87.6251921424, 41.8788655841), Point)",41.857183858,-87.620334624,"List(List(-87.6203346241, 41.8571838585), Point)",2025,3,2026-05-14T05:45:29.752Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=03/part_1230000_50789e9a.parquet
4266df3d65f25a69ce7226f3ee7e2a841eb357a3,19a89fbfe54b0564c6961e347f5faa1158413d94d42ac5f92d09417a75c4362f684b96c26fa660d1171cd110800d2b4e7c059ac469868867dd4ddda0844a8672,2025-03-31T23:00:00.000,2025-03-31T23:00:00.000,13,0.07,null,null,27,27,35,7.1,0,0,42.6,Credit Card,Taxicab Insurance Agency Llc,41.878914496,-87.70589713,"List(List(-87.7058971305, 41.8789144956), Point)",41.878914496,-87.70589713,"List(List(-87.7058971305, 41.8789144956), Point)",2025,3,2026-05-14T05:45:29.752Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=03/part_1230000_50789e9a.parquet
54b7ada7c9729cb0e54eb33f8268b7c930d62eea,a0f44bc0a273e49230e4abe4d0a8d3a1e8305945f8fa0b0556bf2e6410a1c80ab39b5bacd926445645a577a13d52988ce42ef904e3a262c18f5c8c3eb3b901a2,2025-03-31T23:00:00.000,2025-03-31T23:00:00.000,404,1.49,null,null,8,32,9.48,0,0,0,9.98,Mobile,Globe Taxi,41.899602111,-87.633308037,"List(List(-87.6333080367, 41.899602111), Point)",41.878865584,-87.625192142,"List(List(-87.6251921424, 41.8788655841), Point)",2025,3,2026-05-14T05:45:29.752Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2025/month=03/part_1230000_50789e9a.parquet


In [0]:
display(spark.sql(f"SELECT * FROM {bronze_taxi_2026} ORDER BY ingestion_timestamp DESC LIMIT 5"))

trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract,year,month,ingestion_timestamp,source_file
08938026743365ffd454461503704222719f7f98,6f3c8bee0eb9f2eb5b7bd4c279792dbf7c2e1bd5f5c6db017087b4ee58875dc8552fb74f6345eb0e21afa044814a63c14079116be3fdd3708eea8d28d6b52f47,2026-01-31T23:45:00.000,2026-01-31T23:45:00.000,417,1.9,32,8,8,3,0,0,11.5,Credit Card,Flash Cab,41.878865584,-87.625192142,"List(List(-87.6251921424, 41.8788655841), Point)",41.899602111,-87.633308037,"List(List(-87.6333080367, 41.899602111), Point)",null,null,2026,1,2026-05-12T08:13:29.702Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2026/month=01/part_1020000_01445621.parquet
049848083a90085ad7972b406ab4cdce28a4cc2d,6c87f1d023a9d7146ae81b6aa4648bf98cec52cceb9f0854f80a246c81d94ffae05dd954ea58f3a3e82f2bc4793dc9cd826a6c3fdd5af3640dae2c2fe06e387e,2026-01-31T23:45:00.000,2026-02-01T00:15:00.000,1620,0,76,8,56,5.6,0,0,61.6,Credit Card,Transit Administrative Center Inc,41.980264315,-87.913624596,"List(List(-87.913624596, 41.9802643146), Point)",41.899602111,-87.633308037,"List(List(-87.6333080367, 41.899602111), Point)",null,null,2026,1,2026-05-12T08:13:29.702Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2026/month=01/part_1020000_01445621.parquet
08eb5322e59ac09e024d850fc94b2596f4ed9e41,b7ac477e614f1f222f42c698e3f2841fb020060e62917217509f0d03770fa74a0684e6ca49d99a133b11b3fc7bc4ba058ee35d70c6a8c170454d1e05c889d202,2026-01-31T23:45:00.000,2026-02-01T00:00:00.000,1083,16.1,76,null,39.75,10,0,29,79.25,Credit Card,Taxicab Insurance Agency Llc,41.97907082,-87.903039661,"List(List(-87.9030396611, 41.9790708201), Point)",null,null,null,17031980000,null,2026,1,2026-05-12T08:13:29.702Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2026/month=01/part_1020000_01445621.parquet
02a6a858d7e44c62c3c398a87b7994f61a7779d9,2c2846e06a06b51bf72e041bce82b95d5471941476ff25f9dc57a53f1be89850bab3ff974c911e1eb13e2be3392cd74ab84cef84c380c94a4b6f4658744904d7,2026-01-31T23:45:00.000,2026-01-31T23:45:00.000,7,0,3,3,40,8.1,0,0,48.6,Credit Card,Taxicab Insurance Agency Llc,41.96581197,-87.655878786,"List(List(-87.6558787862, 41.96581197), Point)",41.96581197,-87.655878786,"List(List(-87.6558787862, 41.96581197), Point)",null,null,2026,1,2026-05-12T08:13:29.702Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2026/month=01/part_1020000_01445621.parquet
09770b945fc74b2e143993eebacd958250b0f3c0,89f75adc47573b24826e52e49553465713812aacc951235e248a8a52d5e5c23b4f3fea941b976a8b1c501d0e02c337e13ffc449dfd14d52ea460815e14351bb4,2026-01-31T23:45:00.000,2026-02-01T00:15:00.000,1687,18.95,76,33,47,10.6,0,5.5,63.6,Credit Card,Taxicab Insurance Agency Llc,41.980264315,-87.913624596,"List(List(-87.913624596, 41.9802643146), Point)",41.857183858,-87.620334624,"List(List(-87.6203346241, 41.8571838585), Point)",null,null,2026,1,2026-05-12T08:13:29.702Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_taxi_year=2026/month=01/part_1020000_01445621.parquet
